# 7 - The Five-Model Ladder and the Central Comparison

**Covers:** Table 5 (nine trained models), Section 4.9.1 (verification), Section 4.9.3(A) (predictive performance, bootstrap CI on ΔR2, Table 11 diagnostics), plus the per-genre and per-popularity-band reporting implied by Section 4.1.1 and Table 1.

| Model | Features | Under |
|---|---|---|
| Baseline | none; predicts the training-set mean | shared (algorithm-agnostic) |
| Audio-Only | twelve audio descriptors (key one-hot) | XGB + RF |
| Lyric-Only | VADER sentiment | XGB + RF |
| **Control** | audio + lyric | XGB + RF |
| **Alignment-Augmented** | audio + lyric + alignment gap | XGB + RF |

The study's claim rests on **Control vs Alignment-Augmented**, which differ by exactly one column.

**Produces:** `artifacts/models/*.joblib`, `artifacts/test_predictions.csv`, `artifacts/ladder_metrics.csv`, `artifacts/central_comparison.json`, `artifacts/metrics_by_genre.csv`, `artifacts/metrics_by_band.csv`.

In [ ]:
import json
import os
import sys
import time

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

sys.path.insert(0, os.path.abspath('.'))
import modeling_config as mc

pd.set_option('display.width', 170)
N_JOBS = -1

df = mc.load_corpus()
splits = pd.read_csv(mc.artifact('splits.csv'))
manifest = mc.load_json('manifest.json')
config_xgb = mc.load_json('config_xgb.json')
config_rf = mc.load_json('config_rf.json')

df = df.merge(splits[['id', 'partition']], on='id', how='inner')
df = df[df['partition'].isin(['train', 'val', 'test'])].reset_index(drop=True)

X_all, blocks = mc.build_feature_frame(df)
mc.assert_clean(X_all)
y = df[mc.TARGET].astype(float)
part = df['partition']

idx = {p: (part == p).to_numpy() for p in ['train', 'val', 'test']}
print({p: int(m.sum()) for p, m in idx.items()})
print('candidate feature columns:', X_all.shape[1])

## Section 4.9.1 - Verification, before any performance figure is reported

Feature-configuration isolation, manifest reconciliation, and configuration parity. Section 4.9.1: *"If the Control model's matrix contained the alignment column, the study's central comparison would be measuring nothing at all."*

In [ ]:
ladder_cols = {name: mc.columns_for(name, blocks) for name in mc.LADDER}
for name, cols in ladder_cols.items():
    print('%-22s %3d columns' % (name, len(cols)))

# 1. feature-configuration isolation
assert ladder_cols['baseline'] == []
assert 'alignment_gap' not in ladder_cols['control'], 'Control contains the alignment column'
assert 'alignment_gap' in ladder_cols['alignment_augmented']
diff = set(ladder_cols['alignment_augmented']) - set(ladder_cols['control'])
assert diff == {'alignment_gap'}, f'the central pair differs by more than one column: {diff}'
assert set(ladder_cols['control']) == set(ladder_cols['audio_only']) | set(ladder_cols['lyric_only'])
print('\ncentral pair differs by exactly:', diff)

# 2. no forbidden column anywhere
for name, cols in ladder_cols.items():
    assert not (set(cols) & set(mc.FORBIDDEN_COLS)), name

# 3. manifest reconciliation
assert int(manifest['modelled_rows']) == len(df), (manifest['modelled_rows'], len(df))
for p in ['train', 'val', 'test']:
    assert int(manifest['partition_counts'][p]) == int(idx[p].sum()), p
print('manifest reconciliation: PASS')

print('\nSection 4.9.1 feature-configuration isolation: PASS')

## Train the nine models

Identical splits, identical frozen configuration and identical seed within each algorithm; only the feature columns change.

In [ ]:
os.makedirs(mc.artifact('models'), exist_ok=True)

def make_xgb(n_features):
    params = dict(config_xgb['params'])
    return xgb.XGBRegressor(
        eval_metric='rmse',
        early_stopping_rounds=config_xgb['early_stopping_rounds'],
        n_jobs=N_JOBS,
        **params,
    )

def make_rf(n_features):
    p = dict(config_rf['params'])
    return RandomForestRegressor(
        n_estimators=p['n_estimators'],
        min_samples_leaf=p['min_samples_leaf'],
        max_depth=None,
        max_features=max(1, n_features // 3),   # Breiman's p/3 rule for regression
        random_state=mc.RANDOM_SEED,
        n_jobs=N_JOBS,
    )

preds = pd.DataFrame({'id': df.loc[idx['test'], 'id'].to_numpy(),
                      'y_true': y[idx['test']].to_numpy()})
records = []

# --- Baseline: no learned parameters, computed once, shared by both algorithms
train_mean = float(y[idx['train']].mean())
preds['baseline'] = train_mean
records.append({'algorithm': 'shared', 'model': 'baseline', 'n_features': 0,
                'seconds': 0.0, **mc.regression_metrics(preds['y_true'], preds['baseline'])})
print('baseline predicts the training-set mean popularity = %.4f' % train_mean)

for algo, factory in [('xgboost', make_xgb), ('random_forest', make_rf)]:
    for name in mc.ALGO_MODELS:
        cols = ladder_cols[name]
        t0 = time.time()
        model = factory(len(cols))
        Xtr, Xva = X_all.loc[idx['train'], cols], X_all.loc[idx['val'], cols]
        if algo == 'xgboost':
            model.fit(Xtr, y[idx['train']], eval_set=[(Xva, y[idx['val']])], verbose=False)
        else:
            model.fit(Xtr, y[idx['train']])
        yhat = model.predict(X_all.loc[idx['test'], cols])
        key = f'{algo}__{name}'
        preds[key] = yhat
        m = mc.regression_metrics(preds['y_true'], yhat)
        records.append({'algorithm': algo, 'model': name, 'n_features': len(cols),
                        'seconds': round(time.time() - t0, 1), **m})
        joblib.dump(model, mc.artifact('models', f'{key}.joblib'))
        print('%-14s %-22s R2 %+0.4f  RMSE %6.3f  MAE %6.3f  (%.1fs)'
              % (algo, name, m['r2'], m['rmse'], m['mae'], records[-1]['seconds']))

ladder = pd.DataFrame(records)
preds.to_csv(mc.artifact('test_predictions.csv'), index=False)
ladder.to_csv(mc.artifact('ladder_metrics.csv'), index=False)
print()
print(ladder.round(4).to_string(index=False))

## Configuration parity (Section 4.9.1)

Compare the serialized configuration of every trained model within an algorithm. They must differ only in feature count.

In [ ]:
IGNORE = {'max_features', 'n_features_in_', 'feature_names_in_', 'n_jobs', 'feature_types'}

for algo in ['xgboost', 'random_forest']:
    serialized = {}
    for name in mc.ALGO_MODELS:
        model = joblib.load(mc.artifact('models', f'{algo}__{name}.joblib'))
        params = {k: v for k, v in model.get_params().items() if k not in IGNORE}
        serialized[name] = json.dumps(params, sort_keys=True, default=str)
    distinct = set(serialized.values())
    print('%-14s distinct configurations across its four models: %d' % (algo, len(distinct)))
    assert len(distinct) == 1, f'{algo} configuration parity violated'

print('\nSection 4.9.1 configuration parity: PASS')
print('(max_features is excluded from the comparison: Breiman\'s p/3 rule is a')
print(' function of the feature count, not an independently tuned setting.)')

## Table 11 diagnostics

Two thresholds follow from the expected ranges: a Control R2 below 0.05 means the content features carry almost no signal and the alignment comparison cannot be interpreted; substantially above 0.20 means re-verify the split.

In [ ]:
def pick(algo, name):
    row = ladder[(ladder['algorithm'] == algo) & (ladder['model'] == name)].iloc[0]
    return row

base = pick('shared', 'baseline')
ctrl = pick('xgboost', 'control')
aug = pick('xgboost', 'alignment_augmented')
sigma_test = float(preds['y_true'].std())

print('Baseline  R2 %+.4f   (expected ~0.00, slightly negative is normal)' % base['r2'])
print('Control   R2 %+.4f   RMSE %.3f   (expected R2 0.05-0.20)' % (ctrl['r2'], ctrl['rmse']))
print('test-set popularity sigma: %.3f -> RMSE consistent with that R2 band: %.2f-%.2f'
      % (sigma_test, sigma_test * np.sqrt(0.80), sigma_test * np.sqrt(0.95)))

flags = []
if abs(base['r2']) > 0.02:
    flags.append('Baseline R2 is not near zero -- check the mean-predictor and the test set')
if ctrl['r2'] < 0.05:
    flags.append('Control R2 < 0.05 -- Table 11 says investigate the pipeline BEFORE '
                 'drawing any conclusion from the alignment comparison')
if ctrl['r2'] > 0.20:
    flags.append('Control R2 > 0.20 -- Table 11 says re-verify the artist-aware split '
                 'and the exclusion of artist-level metadata (possible leakage)')
print()
print('\n'.join('FLAG: ' + f for f in flags) if flags else 'No Table 11 diagnostic triggered.')

## Section 4.9.3(A) - Paired bootstrap confidence interval on ΔR2

**This is the study's single decision rule.** The test set is resampled with replacement, paired so that each resample scores both models on identical rows; the 2.5th and 97.5th percentiles form the interval. An improvement is established **only** where the interval excludes zero. A nominal gain that does not survive resampling is reported as *no detectable contribution*.

In [ ]:
N_BOOT = 10_000

def paired_bootstrap_delta_r2(y_true, pred_a, pred_b, n_boot=N_BOOT, seed=mc.RANDOM_SEED):
    """ΔR2 = R2(pred_b) - R2(pred_a), recomputed within each paired resample."""
    y_true = np.asarray(y_true, float)
    pred_a = np.asarray(pred_a, float)
    pred_b = np.asarray(pred_b, float)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        take = rng.integers(0, n, n)
        yt = y_true[take]
        ss_tot = np.sum((yt - yt.mean()) ** 2)
        r2a = 1.0 - np.sum((yt - pred_a[take]) ** 2) / ss_tot
        r2b = 1.0 - np.sum((yt - pred_b[take]) ** 2) / ss_tot
        deltas[i] = r2b - r2a
    return deltas

central = {}
for algo in ['xgboost', 'random_forest']:
    a, b = f'{algo}__control', f'{algo}__alignment_augmented'
    r2_c = mc.regression_metrics(preds['y_true'], preds[a])['r2']
    r2_a = mc.regression_metrics(preds['y_true'], preds[b])['r2']
    deltas = paired_bootstrap_delta_r2(preds['y_true'], preds[a], preds[b])
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    excludes_zero = bool(lo > 0 or hi < 0)
    central[algo] = {
        'control_r2': r2_c,
        'alignment_augmented_r2': r2_a,
        'delta_r2': r2_a - r2_c,
        'delta_r2_as_pct_of_control_r2': 100 * (r2_a - r2_c) / r2_c if r2_c else float('nan'),
        'ci_low': float(lo),
        'ci_high': float(hi),
        'ci_excludes_zero': excludes_zero,
        'delta_rmse_points': (mc.regression_metrics(preds['y_true'], preds[b])['rmse']
                              - mc.regression_metrics(preds['y_true'], preds[a])['rmse']),
        'delta_mae_points': (mc.regression_metrics(preds['y_true'], preds[b])['mae']
                             - mc.regression_metrics(preds['y_true'], preds[a])['mae']),
        'n_bootstrap': N_BOOT,
    }
    c = central[algo]
    print('--- %s ---' % algo)
    print('  Control R2            %+.5f' % c['control_r2'])
    print('  Alignment-Augmented   %+.5f' % c['alignment_augmented_r2'])
    print('  ΔR2                   %+.5f  (%.2f%% of Control R2)'
          % (c['delta_r2'], c['delta_r2_as_pct_of_control_r2']))
    print('  95%% bootstrap CI      [%+.5f, %+.5f]' % (c['ci_low'], c['ci_high']))
    print('  ΔRMSE / ΔMAE          %+.4f / %+.4f popularity points'
          % (c['delta_rmse_points'], c['delta_mae_points']))
    print('  VERDICT: %s' % ('contribution established (interval excludes zero)'
                             if excludes_zero else
                             'NO DETECTABLE CONTRIBUTION (interval contains zero)'))
    print()

central['cross_algorithm_sign_agreement'] = bool(
    np.sign(central['xgboost']['delta_r2']) == np.sign(central['random_forest']['delta_r2']))
print('cross-algorithm sign agreement (Section 4.9.2):',
      central['cross_algorithm_sign_agreement'])
print(mc.save_json(central, 'central_comparison.json'))

## Per-genre and per-popularity-band reporting

Section 4.1.1 justifies genre stratification by pointing at per-genre performance reporting, and Table 1 lists metrics stratified by popularity band. Both are produced here.

In [ ]:
test_meta = df.loc[idx['test'], ['id', 'genre', 'popularity']].reset_index(drop=True)
ev = test_meta.merge(preds, on='id')
ev['band'] = pd.cut(ev['popularity'], bins=[0, 20, 40, 60, 80, 100],
                    labels=['1-20', '21-40', '41-60', '61-80', '81-100'])

def breakdown(by):
    out = []
    for key, g in ev.groupby(by, observed=True):
        row = {by: key, 'n': len(g)}
        for algo in ['xgboost', 'random_forest']:
            for name in ['control', 'alignment_augmented']:
                m = mc.regression_metrics(g['y_true'], g[f'{algo}__{name}'])
                row[f'{algo}_{name}_r2'] = m['r2']
                row[f'{algo}_{name}_rmse'] = m['rmse']
            row[f'{algo}_delta_r2'] = (row[f'{algo}_alignment_augmented_r2']
                                       - row[f'{algo}_control_r2'])
        out.append(row)
    return pd.DataFrame(out)

by_genre = breakdown('genre').sort_values('xgboost_delta_r2', ascending=False)
by_band = breakdown('band')

show = ['genre', 'n', 'xgboost_control_r2', 'xgboost_alignment_augmented_r2', 'xgboost_delta_r2']
print('=== by genre ===')
print(by_genre[show].round(4).to_string(index=False))
print()
print('=== by popularity band ===')
print(by_band[[c.replace('genre', 'band') if c == 'genre' else c for c in show]]
      .round(4).to_string(index=False))

by_genre.to_csv(mc.artifact('metrics_by_genre.csv'), index=False)
by_band.to_csv(mc.artifact('metrics_by_band.csv'), index=False)
print('\nsaved metrics_by_genre.csv and metrics_by_band.csv')
print('Note: per-genre R2 is computed against each genre\'s own variance, so')
print('small-n genres will look unstable. Report n alongside every figure.')

---
### How to write this up

- Report **all nine models together** (Section 4.9.3 A) so the result is self-contextualising: Baseline anchors the floor, Audio-Only and Lyric-Only show whether Control is genuinely multimodal or dominated by one side.
- Report ΔR2 **both** absolutely and as a share of Control's own R2, so a detectable but small improvement is characterised as small. The RMSE difference goes in popularity points for the same reason.
- If the interval contains zero, that is a result, not a failure -- Section 4.9.3(A) pre-commits to reporting it as *no detectable contribution*. (NFR1's current wording says the model *shall* achieve an improvement; that wording needs reconciling with this section either way.)

Next: `8_shap.ipynb` for the co-primary result, then `9_robustness.ipynb`.